# HW01-C — Airflow Scheduled Pipeline

A pipeline that only runs when you remember to click it is a chore.

Here you turn the SQL work into an Airflow DAG. The DAG refreshes your materialized view, validates it, and writes a run report.

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

- Airflow DAGs: https://airflow.apache.org/docs/apache-airflow/stable/core-concepts/dags.html
- Airflow Variables: https://airflow.apache.org/docs/apache-airflow/stable/core-concepts/variables.html
- Airflow best practices: https://airflow.apache.org/docs/apache-airflow/stable/best-practices.html

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

In [15]:
from pathlib import Path
import os, re, textwrap

PROJECT = Path.cwd()
for path in ['dags', 'reports', 'screenshots']:
    (PROJECT / path).mkdir(exist_ok=True)

student_id = os.getenv('QBC12_STUDENT_ID', '') or input('GitHub username / student id: ').strip()
safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', student_id.lower())
DAG_ID = f'qbc12_hw01_{safe_student}_airbnb_pipeline'
STUDENT_SCHEMA = f'student_{safe_student}'
DAG_ID, STUDENT_SCHEMA

('qbc12_hw01_mehrad_rafiei_tabatabaei_airbnb_pipeline',
 'student_mehrad_rafiei_tabatabaei')

## 1. DAG design

Build this chain:

```text
read_config → refresh_summary → validate_summary → branch → success/failure report
```

Database credentials must come from Airflow Variables.

In [18]:
# TODO 1.1
# Create dags/<DAG_ID>.py with all tasks.
# Imports, DAG metadata, read_config, refresh, validate, branch, and reports.

import os
from pathlib import Path
from datetime import datetime

# Define DAG_ID again to ensure consistency
student_id = "mehrad_rafiei_tabatabaei"  # your GitHub username
safe_student = student_id.lower().replace(' ', '_')
DAG_ID = f'qbc12_hw01_{safe_student}_airbnb_pipeline'

dag_content = f'''
"""
DAG for refreshing Airbnb materialized view and data validation.
Credentials are read from Airflow Variables.
"""
from datetime import datetime
from pathlib import Path
from airflow import DAG
from airflow.operators.python import PythonOperator, BranchPythonOperator
from airflow.providers.postgres.hooks.postgres import PostgresHook
from airflow.models import Variable
import logging

default_args = {{
    'owner': '{safe_student}',
    'retries': 1,
    'start_date': datetime(2025, 1, 1),
}}

DAG_ID = '{DAG_ID}'
SCHEMA_NAME = 'student_{safe_student}'
MV_NAME = 'mv_airbnb_summary'   
def get_db_config():
    """Fetch PostgreSQL credentials from Airflow Variables."""
    return {{
        'host': Variable.get('mehrad_db_host'),
        'port': Variable.get('mehrad_db_port'),
        'database': Variable.get('mehrad_db_name'),
        'user': Variable.get('mehrad_db_user'),
        'password': Variable.get('mehrad_db_password'),
    }}

def read_config(**context):
    """Read DB config and push to XCom."""
    config = get_db_config()
    config['schema'] = SCHEMA_NAME
    config['mv_name'] = MV_NAME
    context['ti'].xcom_push(key='db_config', value=config)
    logging.info("Config read from Variables")
    return config

def refresh_summary(**context):
    """Refresh materialized view directly in PostgreSQL."""
    ti = context['ti']
    config = ti.xcom_pull(task_ids='read_config', key='db_config')
    hook = PostgresHook(
        postgres_conn_id='postgres_default',
        host=config['host'],
        port=config['port'],
        dbname=config['database'],
        user=config['user'],
        password=config['password'],
        schema=config['schema'],
    )
    sql = f"REFRESH MATERIALIZED VIEW CONCURRENTLY {{config['schema']}}.{{config['mv_name']}};"
    hook.run(sql)
    logging.info(f"Refreshed {{config['schema']}}.{{config['mv_name']}}")
    return {{'refreshed_at': str(datetime.utcnow()), 'mv': config['mv_name']}}

def validate_summary(**context):
    """Run validation checks and return pass/fail status."""
    ti = context['ti']
    config = ti.xcom_pull(task_ids='read_config', key='db_config')
    hook = PostgresHook(
        postgres_conn_id='postgres_default',
        host=config['host'],
        port=config['port'],
        dbname=config['database'],
        user=config['user'],
        password=config['password'],
        schema=config['schema'],
    )
    checks = {{}}
    
    # row_count > 0
    sql = f"SELECT COUNT(*) FROM {{config['schema']}}.{{config['mv_name']}};"
    row_count = hook.get_first(sql)[0]
    checks['row_count_positive'] = row_count > 0
    
    # null_neighbourhoods == 0
    sql = f"SELECT COUNT(*) FROM {{config['schema']}}.{{config['mv_name']}} WHERE neighbourhood IS NULL;"
    null_neigh = hook.get_first(sql)[0]
    checks['null_neighbourhoods_zero'] = (null_neigh == 0)
    
    # bad_prices == 0 (price <= 0 or NULL)
    sql = f"SELECT COUNT(*) FROM {{config['schema']}}.{{config['mv_name']}} WHERE price <= 0 OR price IS NULL;"
    bad_prices = hook.get_first(sql)[0]
    checks['bad_prices_zero'] = (bad_prices == 0)
    
    # bad_availability == 0 (availability_365 not in [0,365])
    sql = f"SELECT COUNT(*) FROM {{config['schema']}}.{{config['mv_name']}} WHERE availability_365 < 0 OR availability_365 > 365 OR availability_365 IS NULL;"
    bad_avail = hook.get_first(sql)[0]
    checks['bad_availability_zero'] = (bad_avail == 0)
    
    all_passed = all(checks.values())
    result = {{'passed': all_passed, 'details': checks, 'row_count': row_count}}
    ti.xcom_push(key='validation_result', value=result)
    logging.info(f"Validation result: {{result}}")
    return result

def choose_report_path(**context):
    """Branch to success or failure based on validation."""
    ti = context['ti']
    val_result = ti.xcom_pull(task_ids='validate_summary', key='validation_result')
    if val_result and val_result.get('passed'):
        return 'write_success_report'
    else:
        return 'write_failure_report'

def write_success_report(**context):
    """Write success report to reports/ directory."""
    ti = context['ti']
    config = ti.xcom_pull(task_ids='read_config', key='db_config')
    val_result = ti.xcom_pull(task_ids='validate_summary', key='validation_result')
    report_path = Path('/opt/airflow/reports/hw01_c_airflow.md')
    content = f"""# Airflow Pipeline Report (Success)
- DAG ID: {{DAG_ID}}
- Run timestamp: {{datetime.utcnow()}}
- Materialized view: {{config['schema']}}.{{config['mv_name']}}
- Validation passed: True
- Row count: {{val_result.get('row_count')}}
- Check details: {{val_result.get('details')}}
"""
    report_path.parent.mkdir(exist_ok=True)
    report_path.write_text(content)
    logging.info(f"Success report written to {{report_path}}")

def write_failure_report(**context):
    """Write failure report then raise ValueError."""
    ti = context['ti']
    config = ti.xcom_pull(task_ids='read_config', key='db_config')
    val_result = ti.xcom_pull(task_ids='validate_summary', key='validation_result')
    report_path = Path('/opt/airflow/reports/hw01_c_airflow.md')
    failed = {{k:v for k,v in val_result.get('details',{{}}).items() if not v}}
    content = f"""# Airflow Pipeline Report (FAILURE)
- DAG ID: {{DAG_ID}}
- Run timestamp: {{datetime.utcnow()}}
- Materialized view: {{config['schema']}}.{{config['mv_name']}}
- Validation passed: False
- Failed checks: {{failed}}
"""
    report_path.parent.mkdir(exist_ok=True)
    report_path.write_text(content)
    logging.error("Validation failed, raising exception")
    raise ValueError("Data validation failed. See report.")

with DAG(
    dag_id=DAG_ID,
    default_args=default_args,
    schedule=None,
    catchup=False,
    tags=['hw01', 'airbnb'],
) as dag:
    task_read_config = PythonOperator(task_id='read_config', python_callable=read_config)
    task_refresh = PythonOperator(task_id='refresh_summary', python_callable=refresh_summary)
    task_validate = PythonOperator(task_id='validate_summary', python_callable=validate_summary)
    task_branch = BranchPythonOperator(task_id='choose_report_path', python_callable=choose_report_path)
    task_success = PythonOperator(task_id='write_success_report', python_callable=write_success_report)
    task_failure = PythonOperator(task_id='write_failure_report', python_callable=write_failure_report)

    task_read_config >> task_refresh >> task_validate >> task_branch
    task_branch >> task_success
    task_branch >> task_failure
'''

# Write the DAG file into the dags/ folder
dag_path = Path('dags') / f'{DAG_ID}.py'
dag_path.parent.mkdir(exist_ok=True)
dag_path.write_text(dag_content)
print(f"DAG file created: {dag_path}")

DAG file created: dags/qbc12_hw01_mehrad_rafiei_tabatabaei_airbnb_pipeline.py


## 2. Refresh task

The refresh task should recreate your materialized view in Postgres. Do not move the full dataset into Python.

In [9]:
# TODO 2.1
# Add refresh_summary(config).
# It should create/refresh your materialized view and indexes.
# Return a small dict, not a dataframe.

# Update your DAG file.

## 3. Validation task

Required checks:

- row_count > 0
- null_neighbourhoods == 0
- bad_prices == 0
- bad_availability == 0

In [10]:
# TODO 3.1
# Add validate_summary(config).
# Return a dict containing each check and passed=True/False.

# Update your DAG file.

## 4. Branching and reports

Success and failure should not look the same.

Use `@task.branch` to choose the report path.

In [11]:
# TODO 4.1
# Add choose_report_path, write_success_report, and write_failure_report.
# Failure report should raise ValueError after writing the report.

# Update your DAG file.

In [12]:
# Syntax check. This is not a full Airflow run.
import py_compile

dag_path = PROJECT / 'dags' / f'{DAG_ID}.py'
assert dag_path.exists(), f'Missing DAG file: {dag_path}'
py_compile.compile(str(dag_path), doraise=True)
print('DAG compiles:', dag_path)

DAG compiles: /home/mehrad/Desktop/project3/HW01_C/dags/qbc12_hw01_mehrad_rafiei_tabatabaei_airbnb_pipeline.py


## 5. Shared Airflow run

In shared Airflow:

1. find your DAG
2. unpause it
3. trigger it manually
4. inspect Graph view
5. inspect logs
6. confirm the materialized view was refreshed

Screenshots:

```text
screenshots/airflow_dag_graph.png
screenshots/airflow_success_run.png
```

In [2]:
# TODO 5.1
# Write reports/hw01_c_airflow.md.
# Include DAG id, Airflow URL, successful run timestamp,
# refreshed object name, validation result, screenshot paths.

from pathlib import Path
from datetime import datetime

report_content = f"""# HW01-C Airflow Pipeline Report

- **DAG ID:** qbc12_hw01_mehrad_rafiei_tabatabaei_airbnb_pipeline
- **Airflow URL:** http://185.50.38.163:33013
- **Successful run timestamp:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
- **Refreshed object:** student_mehrad_rafiei_tabatabaei.mv_airbnb_summary
- **Validation result:** PASSED (all checks true)

## Validation details
- row_count > 0 : True
- null_neighbourhoods == 0 : True
- bad_prices == 0 : True
- bad_availability == 0 : True

"""

Path('reports/hw01_c_airflow.md').write_text(report_content)
print("Report written to reports/hw01_c_airflow.md")

Report written to reports/hw01_c_airflow.md


In [14]:
for file in [f'dags/{DAG_ID}.py', 'reports/hw01_c_airflow.md']:
    assert Path(file).exists(), f'Missing {file}'
print('Basic deliverables exist.')

Basic deliverables exist.


## Commit

```bash
git add dags reports screenshots notebooks
git commit -m "HW01-C Airflow scheduled pipeline"
```